# 02 · From Zarr chunks to SQL rows

**Time:** 35 minutes · **Audience:** undergraduate / introductory graduate GIS and Earth science.

**By the end:** Inspect Zarr metadata, reconstruct indices, and compare equivalent array and SQL filters.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-zarr-sql-views/lite/lab/index.html?path=02_zarr_sql.ipynb) · [Earth lab](https://jltobias.github.io/JupyterLite-zarr-sql-views/lab/) · [Sources and licenses](https://jltobias.github.io/JupyterLite-zarr-sql-views/book/credits.html)

Run cells from top to bottom with **Shift+Enter**. The first kernel start downloads Python WebAssembly packages.
The bundled exercises need no data-service account. Save a copy with **File → Save and Export Notebook As** before clearing browser storage.

![Four-stage workflow: STAC discovery, Zarr reading, SQL selection, linked views](assets/pipeline.svg)

In [1]:
from course import *
import numpy as np
import matplotlib.pyplot as plt
print('Ready: Python + NumPy + local teaching data')

Ready: Python + NumPy + local teaching data


## 1. Inspect the storage contract

Zarr separates metadata from chunk bytes. Our deliberately small stores use Zarr v2, little-endian float32, C order,
no compression inside the store, and one time slice per chunk. The ZIP only packages files for JupyterLite.
The helper is an educational reader for this exact layout; it is not a general Zarr implementation.

In [2]:
with zipfile.ZipFile('data/ngami.zip') as archive:
    spec = json.loads(archive.read('value/.zarray'))
print(json.dumps(spec, indent=2))
print('Bytes per uncompressed chunk:', np.prod(spec['chunks']) * 4)

{
  "zarr_format": 2,
  "shape": [
    3,
    48,
    48
  ],
  "chunks": [
    1,
    48,
    48
  ],
  "dtype": "<f4",
  "compressor": null,
  "fill_value": "NaN",
  "order": "C",
  "filters": null
}
Bytes per uncompressed chunk: 9216


## 2. Flatten without losing geography

For a C-order `(time, y, x)` array: `id = (time * ny + y) * nx + x`.
The latitude coordinates decrease down the array. Never assume row zero is the south edge.

In [3]:
meta, values = load_cube('ngami')
nt, ny, nx = values.shape
t, y, x = 1, 12, 20
cell_id = (t * ny + y) * nx + x
assert values.ravel()[cell_id] == values[t, y, x] or np.isnan(values[t, y, x])
print('id:', cell_id, 'lon:', meta['lon'][x], 'lat:', meta['lat'][y])

id: 2900 lon: 22.719583333333333 lat: -20.41510416666667


## 3. Make an explicit SQL view of the loaded window

SQL runs **after** this bounded cube is decoded. A `WHERE` expression does not automatically reduce remote Zarr reads.
NumPy stores missing values as NaN; the SQL table stores them as NULL. `COUNT(value)` excludes NULL.
SQLite is used in this Python cell. The companion browser lab uses real DuckDB-WASM.

In [4]:
con = sql_connection(meta, values)
query = '''SELECT t, COUNT(value), AVG(value) FROM cells
           WHERE value > 0.5 GROUP BY t ORDER BY t'''
for row in con.execute(query):
    print(row)
sql_n = con.execute('SELECT COUNT(*) FROM cells WHERE value > 0.5 AND t = 0').fetchone()[0]
assert sql_n == np.count_nonzero(values[0] > 0.5)

(0, 127, 0.8186417855615691)


## 4. Experiment with the memory budget

This estimates only the raw float array. SQL tables, Arrow buffers, rendering buffers, and Python objects add overhead.
The lab imposes a 100,000-cell input limit. For remote large cubes, choose a bounding box, variable, resolution, and time window **before** decoding.

In [5]:
for shape in [(3,48,48), (12,256,256), (365,2000,2000)]:
    print(shape, 'raw float32 MiB:', round(np.prod(shape)*4/1024**2, 2))

(3, 48, 48) raw float32 MiB: 0.03
(12, 256, 256) raw float32 MiB: 3.0
(365, 2000, 2000) raw float32 MiB: 5569.46


In [6]:
lab('ngami','cube')

## Try it yourself

Write SQL for a longitude window and two time slices. Verify the count with NumPy. Predict whether it changes bytes already fetched.

## Check your reasoning

The filters should agree once NULL and NaN are handled consistently. Filtering an already-loaded table does not undo network transfer.

## Evidence journal

Write four sentences: **what I observed; how I calculated it; what this cannot establish; what evidence I need next**.
For any figure you reuse, retain its units, date or scenario, data origin, transformations, and attribution.

## Sources and credit

- [zarr-sql-views, source architecture and ISC license](https://github.com/dzole0311/zarr-sql-views)
- [Zarr v2 specification](https://zarr-specs.readthedocs.io/en/latest/v2/v2.0.html)

Teaching adaptation of [dzole0311/zarr-sql-views](https://github.com/dzole0311/zarr-sql-views), ISC.
Original teaching code and prose: JupyterLite-zarr-sql-views contributors, ISC. Data keep their separate licenses.